In [1]:
!pip3 install numpy


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python3.13 -m pip install --upgrade pip
error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this er

In [ ]:
def read_mol_file(mol_filename):
    """ Reads a .mol file and extracts atomic data. """
    atoms = []
    with open(mol_filename, 'r') as file:
        lines = file.readlines()
        
    
    # Find the start of atomic coordinates (skip the first few header lines)
    start_index = 2  # Assuming first few lines are headers
    while not lines[start_index].strip()[0].isdigit():  
        start_index += 1  # Find the first line with numerical values
    
    for line in lines[start_index:]:
        parts = line.split()
        if len(parts) < 6:
            continue
        atom_label = parts[4] + parts[0]  # Example: C1, O2
        x, y, z = map(float, parts[1:4])
        atoms.append((atom_label, parts[4], x, y, z))

    return atoms

def cartesian_to_fractional(cart_coords, lattice_vectors):
    """ Converts Cartesian coordinates to fractional coordinates. """
    lattice_matrix = np.array(lattice_vectors)
    inv_lattice_matrix = np.linalg.inv(lattice_matrix)
    
    cart_matrix = np.array(cart_coords).T  # Convert to a matrix
    fractional_coords = inv_lattice_matrix @ cart_matrix  # Matrix multiplication
    return fractional_coords.T  # Transpose back

def write_cif_file(cif_filename, atoms, lattice_vectors):
    """ Writes atomic data into a .cif file. """
    with open(cif_filename, 'w') as file:
        file.write("_citation_year       2010\n")
        file.write(f"_cell_length_a       {lattice_vectors[0][0]:.3f}\n")
        file.write(f"_cell_length_b       {lattice_vectors[1][1]:.3f}\n")
        file.write(f"_cell_length_c       {lattice_vectors[2][2]:.3f}\n")
        file.write(f"_cell_angle_alpha    90\n")
        file.write(f"_cell_angle_beta     90\n")
        file.write(f"_cell_angle_gamma    90\n")
        file.write(f"_cell_volume         {np.linalg.det(lattice_vectors):.0f}\n")
        file.write("\n_symmetry_cell_setting    triclinic\n")
        file.write("_symmetry_space_group_name_Hall 'P 1'\n")
        file.write("_symmetry_space_group_name_H-M 'P 1'\n")
        file.write("_symmetry_Int_Tables_number 1\n")
        file.write("_symmetry_equiv_pos_as_xyz 'x,y,z'\n\n")
        file.write("loop_\n")
        file.write("_atom_site_label\n")
        file.write("_atom_site_type_symbol\n")
        file.write("_atom_site_fract_x\n")
        file.write("_atom_site_fract_y\n")
        file.write("_atom_site_fract_z\n")
        file.write("_atom_site_charge\n")
        
        # Convert Cartesian to Fractional coordinates
        cartesian_coords = [(x, y, z) for _, _, x, y, z in atoms]
        fractional_coords = cartesian_to_fractional(cartesian_coords, lattice_vectors)
        
        for i, (atom_label, atom_type, _, _, _) in enumerate(atoms):
            fx, fy, fz = fractional_coords[i]
            file.write(f"{atom_label}  {atom_type}  {fx:.5f}  {fy:.5f}  {fz:.5f}  0\n")

def mol_to_cif(mol_filename, cif_filename, lattice_vectors):
    """ Converts a .mol file to a .cif file. """
    atoms = read_mol_file(mol_filename)
    write_cif_file(cif_filename, atoms, lattice_vectors)
    print(f"Conversion successful: {mol_filename} → {cif_filename}")


: 